[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB07_Optimizacion_Interoperabilidad_ONNX.ipynb)
# NB07 — Optimización, interoperabilidad y ONNX

**Continuidad NB04 → NB07 → NB08.** Usaremos la CNN de **piedra, papel o tijera** (128 × 128, píxeles en [0, 1], categorías `paper`, `rock`, `scissors`). La optimización con TFLite y la demostración de poda son experimentos; el archivo ONNX exporta el **modelo Keras original**, sin poda.


## 1. Preparación del entorno


In [ ]:
!pip -q install tf2onnx onnx onnxruntime

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import onnx
import onnxruntime as ort

from tensorflow import keras
from pathlib import Path

print("TensorFlow:", tf.__version__)
print("ONNX Runtime:", ort.__version__)


## 2. Conectar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Recuperar la CNN de piedra, papel o tijera del NB04

En NB04 entrenamos una CNN, pero **el notebook no guardaba automáticamente un archivo .keras**. Por eso este paso ofrece dos posibilidades:

- **Opción A:** si guardaste tu CNN del NB04 en Drive, indica la ruta y cárgala.
- **Opción B (reproducible):** si no tienes ese archivo, descarga los tres ZIP docentes desde GitHub y **reconstruye y entrena la misma arquitectura del NB04** durante cinco épocas. No se inventa otro modelo ni se usa MobileNetV2.

Para la opción B se necesita conexión a Internet en Colab. El entrenamiento se ejecuta una vez y guarda el resultado en Drive para reutilizarlo.

In [ ]:
from tensorflow.keras import layers
from zipfile import ZipFile
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split

CLASS_NAMES = ["paper", "rock", "scissors"]  # Orden exacto del NB04
IMG_SIZE = (128, 128)
SEED = 42
tf.keras.utils.set_random_seed(SEED)
MODEL_PATH = "/content/drive/MyDrive/MachineLearning2026/modelo_cnn_nb04.keras"
Path(MODEL_PATH).parent.mkdir(parents=True, exist_ok=True)

if Path(MODEL_PATH).is_file():
    print("Opción A: cargando modelo guardado del NB04")
    model = keras.models.load_model(MODEL_PATH)
else:
    print("Opción B: reconstruyendo la CNN del NB04 con el dataset docente")
    raiz = Path("/content")
    base_url = ("https://raw.githubusercontent.com/juliopez/"
                "Taller-Fundamentos-Data-Science-Python/main/"
                "Machine_Learning_2026/02_Datasets/dataset_cnn/")
    for clase in ["paper", "rock", "scissors"]:
        nombre = f"dataset_cnn_{clase}.zip"
        zip_local = raiz / nombre
        urlretrieve(base_url + nombre, zip_local)
        with ZipFile(zip_local) as archivo:
            for elemento in archivo.infolist():
                partes = Path(elemento.filename)
                if partes.is_absolute() or ".." in partes.parts or partes.parts[0] != "dataset_cnn":
                    raise ValueError(f"Ruta no válida en ZIP: {elemento.filename}")
            archivo.extractall(raiz)
    train_dir = raiz / "dataset_cnn" / "train"
    rutas, etiquetas = [], []
    for indice, clase in enumerate(CLASS_NAMES):
        archivos = sorted((train_dir / clase).glob("*.png"))
        assert archivos, f"No se encontraron imágenes para {clase}"
        rutas.extend(str(p) for p in archivos)
        etiquetas.extend([indice] * len(archivos))
    rutas_train, rutas_val, etiquetas_train, etiquetas_val = train_test_split(
        rutas, etiquetas, test_size=0.20, random_state=SEED, stratify=etiquetas
    )
    def leer(ruta, etiqueta):
        imagen = tf.io.decode_png(tf.io.read_file(ruta), channels=3)
        imagen = tf.image.resize(imagen, IMG_SIZE)
        return tf.cast(imagen, tf.float32) / 255.0, etiqueta
    def dataset(rutas_, etiquetas_, mezclar=False):
        ds = tf.data.Dataset.from_tensor_slices((rutas_, np.array(etiquetas_, dtype=np.int32)))
        if mezclar:
            ds = ds.shuffle(len(rutas_), seed=SEED, reshuffle_each_iteration=True)
        return ds.map(leer, num_parallel_calls=tf.data.AUTOTUNE).batch(16).prefetch(tf.data.AUTOTUNE)
    model = keras.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.Conv2D(16, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax"),
    ], name="cnn_piedra_papel_tijera")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(dataset(rutas_train, etiquetas_train, True),
              validation_data=dataset(rutas_val, etiquetas_val), epochs=5)
    model.save(MODEL_PATH)
    print("CNN guardada:", MODEL_PATH)

assert tuple(model.input_shape[1:]) == (128, 128, 3), "El modelo no coincide con NB04"
assert model.output_shape[-1] == 3, "Se esperan tres categorías"
model.summary()

## 4. Contrato del modelo: entrada, clases y preprocesamiento

El **contrato** describe exactamente qué espera recibir la CNN. En NB04 cada imagen se convertía a RGB, se redimensionaba a **128 × 128** y sus píxeles se dividían entre **255**. **No** se utilizó `MobileNetV2.preprocess_input`: aplicarlo aquí produciría entradas incompatibles con el entrenamiento.

El orden de salida es **paper, rock, scissors**. ONNX y Streamlit deben conservar este mismo contrato.

In [ ]:
contrato = {
    "input_shape": [1, 128, 128, 3],
    "tipo_entrada": "float32",
    "canales": "RGB",
    "preprocesamiento": "RGB, resize 128x128, float32 / 255.0",
    "clases": CLASS_NAMES
}
print(contrato)

## 5. Seleccionar una imagen de prueba

Utilizaremos una imagen **del conjunto de prueba original del NB04**, diferente de las usadas para entrenar. Si cargaste el modelo guardado (opción A), el dataset se descargará igualmente para disponer de una imagen de ejemplo.

In [ ]:
from PIL import Image
from zipfile import ZipFile
from urllib.request import urlretrieve

TEST_DIR = Path("/content/dataset_cnn/test")
if not TEST_DIR.is_dir():
    raiz = Path("/content")
    base_url = ("https://raw.githubusercontent.com/juliopez/"
                "Taller-Fundamentos-Data-Science-Python/main/"
                "Machine_Learning_2026/02_Datasets/dataset_cnn/")
    for clase in CLASS_NAMES:
        nombre = f"dataset_cnn_{clase}.zip"
        zip_local = raiz / nombre
        urlretrieve(base_url + nombre, zip_local)
        with ZipFile(zip_local) as archivo:
            for elemento in archivo.infolist():
                partes = Path(elemento.filename)
                if partes.is_absolute() or ".." in partes.parts or partes.parts[0] != "dataset_cnn":
                    raise ValueError(f"Ruta no válida: {elemento.filename}")
            archivo.extractall(raiz)
IMAGE_PATH = sorted((TEST_DIR / "rock").glob("*.png"))[0]
img = Image.open(IMAGE_PATH).convert("RGB").resize(IMG_SIZE)
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f"Ejemplo de prueba: {IMAGE_PATH.parent.name}")
plt.axis("off")
plt.show()

## 6. Preprocesar la imagen como en NB04

**RGB → 128 × 128 → float32 → dividir entre 255 → agregar dimensión de lote**. La entrada resultante tendrá forma `(1, 128, 128, 3)`. Usaremos exactamente el **mismo tensor `x`** para Keras, TFLite y ONNX.

In [ ]:
x = np.asarray(img, dtype=np.float32)[None, ...] / 255.0
assert x.shape == (1, 128, 128, 3)
print("Shape:", x.shape, "dtype:", x.dtype)
print("Rango de píxeles:", float(x.min()), float(x.max()))

## 7. Baseline del modelo original

Antes de optimizar o convertir el modelo registraremos:

- tamaño;
- latencia de inferencia;
- predicción;
- confianza.

Estos valores constituyen la **baseline**.


In [ ]:
keras_size_mb = os.path.getsize(MODEL_PATH) / (1024 ** 2)

# Warm-up
for _ in range(5):
    _ = model.predict(x, verbose=0)

tiempos = []

for _ in range(30):
    inicio = time.perf_counter()
    pred = model.predict(x, verbose=0)
    fin = time.perf_counter()
    tiempos.append((fin - inicio) * 1000)

keras_latency = np.mean(tiempos)
keras_prediction = int(np.argmax(pred[0]))
keras_confidence = float(np.max(pred[0]))

print(f"Tamaño modelo Keras: {keras_size_mb:.2f} MB")
print(f"Latencia promedio: {keras_latency:.2f} ms")
print("Clase predicha:", CLASS_NAMES[keras_prediction])
print(f"Confianza: {keras_confidence:.2%}")


## 8. Cuantificación

La cuantificación reduce la precisión numérica utilizada para representar determinados valores del modelo.

En esta práctica generaremos una versión **TensorFlow Lite cuantificada dinámicamente** para observar el efecto sobre el tamaño del archivo.


La cuantificación dinámica puede reducir el tamaño del archivo, pero **no garantiza menor latencia ni mayor exactitud**. La mediremos sin modificar el modelo Keras original.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_quant_model = converter.convert()

TFLITE_PATH = "/content/modelo_quantized.tflite"

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_quant_model)

tflite_size_mb = os.path.getsize(TFLITE_PATH) / (1024 ** 2)

print(f"Tamaño TFLite cuantificado: {tflite_size_mb:.2f} MB")


## 9. Inferencia con el modelo cuantificado


In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Entrada:", input_details[0]["shape"], input_details[0]["dtype"])
print("Salida:", output_details[0]["shape"], output_details[0]["dtype"])


In [ ]:
tflite_input = x.astype(input_details[0]["dtype"])

# Warm-up
for _ in range(5):
    interpreter.set_tensor(input_details[0]["index"], tflite_input)
    interpreter.invoke()

tiempos_tflite = []

for _ in range(30):
    interpreter.set_tensor(input_details[0]["index"], tflite_input)

    inicio = time.perf_counter()
    interpreter.invoke()
    fin = time.perf_counter()

    tiempos_tflite.append((fin - inicio) * 1000)

tflite_pred = interpreter.get_tensor(output_details[0]["index"])

tflite_latency = np.mean(tiempos_tflite)
tflite_prediction = int(np.argmax(tflite_pred[0]))
tflite_confidence = float(np.max(tflite_pred[0]))

print(f"Latencia TFLite: {tflite_latency:.2f} ms")
print("Clase predicha:", CLASS_NAMES[tflite_prediction])
print(f"Confianza: {tflite_confidence:.2%}")


## 10. Comparar modelo original y cuantificado


In [ ]:
comparacion_optimizacion = pd.DataFrame({
    "Modelo": ["Keras original", "TFLite cuantificado"],
    "Tamaño_MB": [keras_size_mb, tflite_size_mb],
    "Latencia_ms": [keras_latency, tflite_latency],
    "Clase_predicha": [
        CLASS_NAMES[keras_prediction],
        CLASS_NAMES[tflite_prediction]
    ],
    "Confianza": [
        keras_confidence,
        tflite_confidence
    ]
})

comparacion_optimizacion


## 11. Poda de redes

La poda (*pruning*) busca reducir conexiones o pesos de baja relevancia.

En esta práctica la abordaremos mediante una demostración sencilla de **poda por magnitud** sobre los pesos del modelo. El objetivo es observar el concepto de **sparsity** sin reemplazar todavía el modelo definitivo.


In [ ]:
pesos = model.get_weights()

total = 0
ceros_originales = 0

for w in pesos:
    total += w.size
    ceros_originales += np.count_nonzero(w == 0)

sparsity_original = ceros_originales / total

print(f"Sparsity original: {sparsity_original:.2%}")


## 12. Simular poda por magnitud


In [ ]:
pesos_podados = []
valores_absolutos = np.concatenate([
    np.abs(w.flatten())
    for w in pesos
    if w.ndim > 1
])

umbral = np.percentile(valores_absolutos, 30)

for w in pesos:
    w_nuevo = w.copy()

    if w.ndim > 1:
        w_nuevo[np.abs(w_nuevo) < umbral] = 0

    pesos_podados.append(w_nuevo)

total_podado = sum(w.size for w in pesos_podados)
ceros_podados = sum(np.count_nonzero(w == 0) for w in pesos_podados)

sparsity_podada = ceros_podados / total_podado

print(f"Umbral aplicado: {umbral:.6f}")
print(f"Sparsity después de la poda: {sparsity_podada:.2%}")


## 13. Crear una copia del modelo con pesos podados

Esta versión se utiliza solamente para comparar el efecto de la poda.


In [ ]:
pruned_model = keras.models.clone_model(model)
pruned_model.set_weights(pesos_podados)

pred_pruned = pruned_model.predict(x, verbose=0)

pruned_prediction = int(np.argmax(pred_pruned[0]))
pruned_confidence = float(np.max(pred_pruned[0]))

print("Clase original:", CLASS_NAMES[keras_prediction])
print("Clase después de poda:", CLASS_NAMES[pruned_prediction])
print(f"Confianza original: {keras_confidence:.2%}")
print(f"Confianza después de poda: {pruned_confidence:.2%}")


## 14. Interpretar la poda

La poda no debe evaluarse solamente sobre una imagen. En el proyecto deberá analizarse utilizando un conjunto de datos y métricas de desempeño.

Observe además que aumentar la cantidad de ceros **no garantiza automáticamente** una reducción equivalente del tamaño del archivo ni una mejora de latencia. El beneficio depende del formato y del runtime utilizado.


## 15. Exportar el modelo a ONNX

ONNX permite representar el modelo en un formato interoperable para ejecutarlo posteriormente mediante runtimes diferentes del framework original.


**Atención:** exportamos la **CNN original del NB04**. Las versiones cuantificada y podada son comparaciones didácticas independientes. Para mantener el contrato, la entrada ONNX será `float32` con forma `(None, 128, 128, 3)` y valores entre 0 y 1.

In [ ]:
import tf2onnx

ONNX_PATH = "/content/modelo_cnn.onnx"
input_signature = (tf.TensorSpec((None, 128, 128, 3), tf.float32, name="input"),)

# tf2onnx.from_keras puede presentar incompatibilidades con ciertas versiones de Keras 3.
# Exportamos primero una función TensorFlow concreta y la convertimos a ONNX.
@tf.function(input_signature=list(input_signature))
def inferencia_tf(entrada):
    return {"probabilidades": model(entrada, training=False)}

onnx_model, _ = tf2onnx.convert.from_function(
    inferencia_tf, input_signature=list(input_signature), opset=13
)
with open(ONNX_PATH, "wb") as archivo:
    archivo.write(onnx_model.SerializeToString())
print("Modelo ONNX generado:", ONNX_PATH)

## 16. Validar el archivo ONNX


In [ ]:
onnx_model_check = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model_check)

onnx_size_mb = os.path.getsize(ONNX_PATH) / (1024 ** 2)

print("Modelo ONNX válido")
print(f"Tamaño: {onnx_size_mb:.2f} MB")


## 17. Cargar ONNX con ONNX Runtime


In [ ]:
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print("Input:", input_name)
print("Input shape:", session.get_inputs()[0].shape)
print("Output:", output_name)
print("Output shape:", session.get_outputs()[0].shape)


## 18. Inferencia mediante ONNX Runtime

El modelo exportado debe recibir exactamente el mismo tipo de entrada que el modelo original.


In [ ]:
onnx_input = x.astype(np.float32)

# Warm-up
for _ in range(5):
    _ = session.run(
        [output_name],
        {input_name: onnx_input}
    )

tiempos_onnx = []

for _ in range(30):
    inicio = time.perf_counter()

    onnx_pred = session.run(
        [output_name],
        {input_name: onnx_input}
    )[0]

    fin = time.perf_counter()
    tiempos_onnx.append((fin - inicio) * 1000)

onnx_latency = np.mean(tiempos_onnx)
onnx_prediction = int(np.argmax(onnx_pred[0]))
onnx_confidence = float(np.max(onnx_pred[0]))

print(f"Latencia ONNX Runtime: {onnx_latency:.2f} ms")
print("Clase predicha:", CLASS_NAMES[onnx_prediction])
print(f"Confianza: {onnx_confidence:.2%}")


## 19. Comparar Keras y ONNX


In [ ]:
comparacion_onnx = pd.DataFrame({
    "Modelo": ["Keras", "ONNX"],
    "Tamaño_MB": [keras_size_mb, onnx_size_mb],
    "Latencia_ms": [keras_latency, onnx_latency],
    "Clase_predicha": [
        CLASS_NAMES[keras_prediction],
        CLASS_NAMES[onnx_prediction]
    ],
    "Confianza": [
        keras_confidence,
        onnx_confidence
    ]
})

comparacion_onnx


## 20. Verificar equivalencia numérica


In [ ]:
diferencia_maxima = np.max(
    np.abs(pred[0] - onnx_pred[0])
)

print(f"Diferencia máxima Keras vs ONNX: {diferencia_maxima:.8f}")


## 21. Guardar los artefactos para NB08

Copiaremos a Google Drive el **modelo ONNX original**, el modelo TFLite de comparación y los metadatos del contrato. En **NB08** utilizaremos `modelo_cnn.onnx` y `class_names.json` para construir la aplicación Streamlit.

In [ ]:
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/MachineLearning2026/implementacion"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import shutil

shutil.copy(ONNX_PATH, OUTPUT_DIR / "modelo_cnn.onnx")
shutil.copy(TFLITE_PATH, OUTPUT_DIR / "modelo_quantized.tflite")

print("Archivos guardados en:")
print(OUTPUT_DIR)


## 22. Guardar el orden de las clases y el contrato de inferencia

No basta con guardar los pesos: la aplicación debe conocer **el orden de las categorías y el preprocesamiento exacto**. Guardaremos ambos metadatos junto al archivo ONNX.

In [ ]:
import json
CLASSES_PATH = OUTPUT_DIR / "class_names.json"
CONTRACT_PATH = OUTPUT_DIR / "model_contract.json"
with open(CLASSES_PATH, "w", encoding="utf-8") as archivo:
    json.dump(CLASS_NAMES, archivo, ensure_ascii=False, indent=2)
with open(CONTRACT_PATH, "w", encoding="utf-8") as archivo:
    json.dump(contrato, archivo, ensure_ascii=False, indent=2)
print("Clases:", CLASSES_PATH)
print("Contrato:", CONTRACT_PATH)

## 23. Comparación final

Complete la tabla con las métricas obtenidas para las versiones que correspondan a su proyecto.


In [ ]:
comparacion_final = pd.DataFrame({
    "Versión": [
        "Keras original",
        "TFLite cuantificado",
        "ONNX"
    ],
    "Tamaño_MB": [
        keras_size_mb,
        tflite_size_mb,
        onnx_size_mb
    ],
    "Latencia_ms": [
        keras_latency,
        tflite_latency,
        onnx_latency
    ],
    "Confianza_imagen_prueba": [
        keras_confidence,
        tflite_confidence,
        onnx_confidence
    ]
})

comparacion_final


## 24. Actividad

A partir de la CNN de piedra, papel o tijera del NB04:

1. Registre tamaño y latencia del modelo original.
2. Defina una baseline antes de optimizar.
3. Genere una versión cuantificada.
4. Compare tamaño, latencia y predicción con el modelo original.
5. Analice conceptualmente la poda y mida la sparsity obtenida.
6. Determine qué efectos podría tener una poda excesiva.
7. Exporte el modelo a ONNX.
8. Valide la estructura del archivo ONNX.
9. Ejecute inferencia mediante ONNX Runtime.
10. Compare la predicción de Keras y ONNX.
11. Compruebe que el contrato de entrada y salida se conserva.
12. Justifique qué versión utilizaría para la implementación del producto final.


## 25. Cierre y continuidad del aprendizaje

Este notebook inicia la etapa técnica del proceso de integración y despliegue.

El resultado esperado es disponer de evidencia sobre eficiencia e interoperabilidad y de un modelo preparado para integrarse posteriormente con una aplicación.

**Modelo definitivo → baseline → cuantificación/poda → comparación → ONNX → ONNX Runtime → modelo preparado para implementación**

En NB08 se utilizarán estos artefactos para construir la aplicación con **Streamlit**, organizar el proyecto en **GitHub** y realizar el despliegue mediante **Streamlit Community Cloud**.
